## Concept 8 — Partial Prompts & Dynamic Templates

### What is it?
Partial prompts let you pre-fill some variables in a template at startup,
leaving others for runtime. This separates concerns: config-time values vs request-time values.

### Pipeline
```
Full template (all variables)
  ↓ partial() at startup
Partial template (some variables fixed)
  ↓ .format() at request time
Final prompt → LLM → Answer
```

### Limitation (what Concept 9 fixes)
Large prompts become unmanageable as one big string.
→ Concept 9 uses PipelinePromptTemplate to compose modular prompt pieces.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from datetime import datetime
from PromptUtils import get_llm, TEST_INPUTS, run_inputs
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm()
print('LLM ready')

### Step 1 — The Problem: Repetitive Variable Filling
Without partial prompts, you repeat the same values every single call.

In [ ]:
base_template = PromptTemplate(
    input_variables=['role', 'date', 'language', 'question'],
    template='You are a {role}. Today is {date}. Respond in {language}. Question: {question}'
)

# Without partial: must supply ALL variables every time
r1 = llm.invoke(base_template.format(
    role='Python expert', date='2024-01-01', language='English', question='What is a list?'
))
# role, date, language are the same every call — wasteful and error-prone
print('Without partial — must repeat role/date/language every call')

### Step 2 — Partial with Static Values
Pre-fill the values known at startup. Only `question` remains for runtime.

In [ ]:
# Partially fill: role, date, language are fixed
partial_template = base_template.partial(
    role='Python expert',
    date='2024-01-15',
    language='English'
)

# Now only 'question' is needed
print(f'Remaining variables: {partial_template.input_variables}')

prompt_str = partial_template.format(question='What is a generator?')
print(f'Final prompt: {prompt_str}')

response = llm.invoke(prompt_str)
print(f'Response: {response.content}')

### Step 3 — Partial with a Function (Dynamic Values)
Pass a callable — it gets evaluated fresh at each `.format()` call.

In [ ]:
# date is computed dynamically on every call
dynamic_template = base_template.partial(
    role='Python expert',
    date=lambda: datetime.now().strftime('%Y-%m-%d'),  # called fresh each time
    language='English'
)

prompt1 = dynamic_template.format(question='What is asyncio?')
print(f'Prompt 1 date: {datetime.now().strftime("%Y-%m-%d")} (embedded live)')

response = llm.invoke(prompt1)
print(f'Response: {response.content[:200]}...')

### Step 4 — Dynamic System Prompt Selection
Choose a different template depending on the context (user type, topic, etc.).

In [ ]:
SYSTEM_CONFIGS = {
    'beginner': {
        'style':      'simple analogies, no jargon',
        'length':     '2-3 short sentences',
    },
    'expert': {
        'style':      'technical precision, correct terminology',
        'length':     'comprehensive but concise',
    },
    'child': {
        'style':      'very simple words, fun examples',
        'length':     '1 sentence',
    },
}

base_chat = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful teacher. Explain using {style}. Keep it {length}.'),
    ('human',  '{question}'),
])

def get_chain_for(user_type: str):
    cfg = SYSTEM_CONFIGS.get(user_type, SYSTEM_CONFIGS['beginner'])
    partial = base_chat.partial(style=cfg['style'], length=cfg['length'])
    return partial | llm | StrOutputParser()

question = 'What is a database index?'
for user_type in ['beginner', 'expert', 'child']:
    chain = get_chain_for(user_type)
    result = chain.invoke({'question': question})
    print(f'\n[{user_type}]\n{result}')
    print('-' * 40)

### Full Function

In [ ]:
# Default: expert-level partial chain
expert_chain = get_chain_for('expert')

def partial_prompt(query):
    return expert_chain.invoke({'question': query})

run_inputs(partial_prompt, label='Concept 8 — Partial Prompts (Expert)')